# ChromaDB Vector Store Explorer

Visualise et explore le contenu de la base de données ChromaDB contenant les documents Terraform.

In [1]:
from pathlib import Path
import pandas as pd
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
import sys

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))
from terraform_agent.config import Config

# Initialize configuration
config = Config(Path.cwd().parent)
vectorstore_path = config.PROJECT_ROOT / "notebooks" / ".vectorstore"

print(f"📁 Looking for vectorstore at: {vectorstore_path}")
print(f"✓ Exists: {vectorstore_path.exists()}")

# Load vectorstore
embedding_function = OllamaEmbeddings(model=config.EMBEDDING_MODEL)
vectorstore = Chroma(
    collection_name="terraform_docs",
    embedding_function=embedding_function,
    persist_directory=str(vectorstore_path),
)

print("✓ Vectorstore loaded successfully")

# Get all documents
all_data = vectorstore.get()
ids = all_data["ids"]
metadatas = all_data["metadatas"]
documents = all_data["documents"]

print(f"📊 Total documents indexed: {len(ids)}")

# Create DataFrame for better visualization
data = []
for i, (doc_id, metadata, content) in enumerate(zip(ids, metadatas, documents)):
    data.append({
        'ID': doc_id,
        'Source': metadata.get('source', 'unknown'),
        'Content Preview': content[:80] + '...' if len(content) > 80 else content,
        'Length': len(content),
    })

df = pd.DataFrame(data)
print(df.to_string())

# Group by source
print("\n📚 Documents by Source:")
print(df.groupby('Source').size())

# Content length statistics
print("\n📈 Content Length Statistics:")
print(df['Length'].describe())

📁 Looking for vectorstore at: /Users/melkouhen/audit-tools/test-langchain/notebooks/.vectorstore
✓ Exists: True
✓ Vectorstore loaded successfully
📊 Total documents indexed: 6
                                     ID                                                         Source                                                                          Content Preview  Length
0  879e7b93-b41e-47e4-836f-90bad58b8803  /Users/melkouhen/audit-tools/test-langchain/docs/structure.md    # Bonnes Pratiques de Structuration Terraform\n\n## 1. Organisation Générale du Pr...     800
1  fc418935-6ccb-4993-b7ae-e8196b852368  /Users/melkouhen/audit-tools/test-langchain/docs/structure.md  ## 3. Gestion des Environnements\n\n| Contexte | Règle |\n|----------|-------|\n| **...     885
2  225fec2b-4493-4974-afbc-717e2fe0f336  /Users/melkouhen/audit-tools/test-langchain/docs/structure.md  ## 5. État et Verrouillage\n\n| Contexte | Règle |\n|----------|-------|\n| **tf-rem...     981
3  f9fbe500-7c33-4b1b-a17

## Search Documents

Recherche sémantique dans la base de données

In [2]:
# Interactive search
query = "module cloud storage"
k = 5

results = vectorstore.similarity_search(query, k=k)

print(f"🔍 Search Results for: '{query}'\n")
for i, result in enumerate(results, 1):
    print(f"{i}. [Source: {result.metadata.get('source', 'unknown')}]")
    print(f"   {result.page_content[:150]}...\n")

🔍 Search Results for: 'module cloud storage'

1. [Source: /Users/melkouhen/audit-tools/test-langchain/docs/structure.md]
   # Bonnes Pratiques de Structuration Terraform

## 1. Organisation Générale du Projet

| Contexte | Règle |
|----------|-------|
| **tf-project-layout*...

2. [Source: /Users/melkouhen/audit-tools/test-langchain/docs/structure.md]
   ## 3. Gestion des Environnements

| Contexte | Règle |
|----------|-------|
| **tf-env-separation** | Préférer la séparation par dossiers (`envs/dev/`...

3. [Source: /Users/melkouhen/audit-tools/test-langchain/docs/structure.md]
   ## Pièges Courants à Éviter

| Contexte | Règle |
|----------|-------|
| **tf-overengineering-modules** | Ne pas wraper une ressource unique (EC2, buc...

4. [Source: /Users/melkouhen/audit-tools/test-langchain/docs/structure.md]
   ## 5. État et Verrouillage

| Contexte | Règle |
|----------|-------|
| **tf-remote-state** | Utiliser un backend distant (S3 + DynamoDB ou Terraform ...

5. [Source: /Users/mel

In [3]:
# Advanced: Find documents about specific topics
topics = [
    "security",
    "modules",
    "variables",
    "state"
]

print("\n🎯 Topic-based Search Results:\n")
for topic in topics:
    results = vectorstore.similarity_search(topic, k=2)
    print(f"\n📌 {topic.upper()}:")
    if results:
        print(f"   {results[0].page_content[:150]}...")
    else:
        print("   No results found")


🎯 Topic-based Search Results:


📌 SECURITY:
   ## 7. Sécurité et Secrets

| Contexte | Règle |
|----------|-------|
| **tf-no-hardcoded-secrets** | Ne jamais encoder les secrets en dur. Utiliser de...

📌 MODULES:
   # Bonnes Pratiques de Structuration Terraform

## 1. Organisation Générale du Projet

| Contexte | Règle |
|----------|-------|
| **tf-project-layout*...

📌 VARIABLES:
   ## 7. Sécurité et Secrets

| Contexte | Règle |
|----------|-------|
| **tf-no-hardcoded-secrets** | Ne jamais encoder les secrets en dur. Utiliser de...

📌 STATE:
   ## 5. État et Verrouillage

| Contexte | Règle |
|----------|-------|
| **tf-remote-state** | Utiliser un backend distant (S3 + DynamoDB ou Terraform ...


In [ ]:
# Summary
print("\n" + "="*60)
print("📊 VECTORSTORE SUMMARY")
print("="*60)
print(f"Total chunks indexed: {len(ids)}")
print(f"Unique source files: {df['Source'].nunique()}")
print(f"Average chunk size: {df['Length'].mean():.0f} characters")
print(f"Embedding model: {config.EMBEDDING_MODEL}")
print(f"Collection: terraform_docs")